# Ring yields 2 — phase-space maps

Mass-action equilibrium yields over (θ, E_AB) at fixed density, from the partition scan of notebook 1. These replace the `Z/ΣZ` maps of `oldSelfAssemblyIG/.../notebook_3_analytical_yields.ipynb`: yields now depend on density through the fugacity z, chains are included, and the monomer weight is 2πA.

Intermediate results are cached: the interpolated scan is cheap; each density's maps are stored in `data/yield_maps/` keyed by the scan contents and options. Set `OVERWRITE=True` to rebuild.

In [ ]:
import os, sys
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / 'base').is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from base.jax_setup import configure_jax
configure_jax()
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False})
from base.paths import PARTITION_DIR, PLOTS_DIR, DATA_DIR, ensure_dir
FIG_DIR = ensure_dir(PLOTS_DIR / 'ring_yields')

In [ ]:
from analysis import yield_maps as ym
OVERWRITE = False
PHIS = [0.05, 0.1, 0.2]           # area fractions
scan = ym.load_scan()
fine = ym.refine(scan)             # 1 deg in theta, 4x in E_AB (PCHIP of ln q)
maps = {phi: ym.cached_maps(fine, phi, overwrite=OVERWRITE) for phi in PHIS}
print(fine.shape, 'grid;', 'maps for phi =', PHIS)

## Dominant ring
Ring size with the largest cluster yield; hatched where rings are under 10% of clusters. Dotted lines: matching angles θ_n.

In [ ]:
fig, axes = plt.subplots(1, len(PHIS), figsize=(5.2 * len(PHIS), 4), constrained_layout=True)
for ax, phi in zip(axes, PHIS):
    ym.plot_dominant_ring(maps[phi], ax=ax, min_yield=0.1)
fig.savefig(FIG_DIR / 'dominant_ring.png', dpi=150); plt.show()

## Relative yields
(Y_a − Y_b)/(Y_a + Y_b) of cluster yields: triangle vs square (the old notebook's main panel), then square vs pentagon and pentagon vs hexagon.

In [ ]:
PAIRS = [('ring_3', 'ring_4'), ('ring_4', 'ring_5'), ('ring_5', 'ring_6')]
phi = 0.2
fig, axes = plt.subplots(1, len(PAIRS), figsize=(5.2 * len(PAIRS), 4), constrained_layout=True)
for ax, (a, b) in zip(axes, PAIRS):
    ym.plot_relative(maps[phi], a, b, ax=ax)
fig.savefig(FIG_DIR / f'relative_yields_phi{phi}.png', dpi=150); plt.show()

### Density dependence
Relative yields of different-size rings move with density as z^{n−m}; the triangle–square boundary shifts with φ.

In [ ]:
fig, axes = plt.subplots(1, len(PHIS), figsize=(5.2 * len(PHIS), 4), constrained_layout=True)
for ax, p in zip(axes, PHIS):
    ym.plot_relative(maps[p], 'ring_3', 'ring_4', ax=ax)
fig.savefig(FIG_DIR / 'triangle_vs_square_vs_density.png', dpi=150); plt.show()

## Per-species yields

In [ ]:
fig = ym.plot_species_grid(maps[phi], labels=['monomer', 'chain_2', 'chain_3', 'chain_4'] +
                           [f'ring_{n}' for n in range(3, 11)], ncols=4)
fig.savefig(FIG_DIR / f'per_species_cluster_phi{phi}.png', dpi=150); plt.show()
fig = ym.plot_species_grid(maps[phi], labels=[f'ring_{n}' for n in range(3, 11)], which='particle', ncols=4)
fig.savefig(FIG_DIR / f'per_species_particle_phi{phi}.png', dpi=150); plt.show()

## Assembly and sampling quality
Assembled fraction (particles not monomers), fraction in rings, and the particle yield carried by species whose ln q failed the ESS/error cut (should be small wherever conclusions are drawn).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4), constrained_layout=True)
ym.plot_assembled(maps[phi], ax=axes[0])
ym.plot_assembled(maps[phi], ax=axes[1], rings_only=True)
ym.plot_unreliable(maps[phi], ax=axes[2])
fig.savefig(FIG_DIR / f'assembled_phi{phi}.png', dpi=150); plt.show()

## Finite N
For N of order 100 the canonical recursion differs from infinite-system mass action. Cut at one E_AB, N = 100 particles at φ.

In [ ]:
from analysis.mass_action import at_density, finite_n, density_from_area_fraction
N = 100
rho = density_from_area_fraction(phi)
i = int(np.argmin(np.abs(scan.eabs - 10)))
fig, ax = plt.subplots(figsize=(7, 4))
for n in range(3, 8):
    inf_, fin_ = [], []
    for j in range(len(scan.thetas_deg)):
        t = scan.table(i, j).finite()
        inf_.append(at_density(t, rho).get(f'ring_{n}', 'particle'))
        fin_.append(finite_n(t, N, N / rho).get(f'ring_{n}', 'particle'))
    c = ym.ring_color(n)
    ax.plot(scan.thetas_deg, inf_, color=c, lw=2, label=f'ring {n}')
    ax.plot(scan.thetas_deg, fin_, color=c, lw=2, ls='--')
ax.set_xlabel('θ (deg)'); ax.set_ylabel('particle yield')
ax.set_title(f'E_AB={scan.eabs[i]:g}, φ={phi}: solid N→∞, dashed N={N}', fontsize=10)
ax.legend(frameon=False, fontsize=8); ax.grid(alpha=0.25, lw=0.5)
fig.savefig(FIG_DIR / 'finite_N.png', dpi=150); plt.show()